<a href="https://colab.research.google.com/github/FernandoCasco/BI-Transporte-AMSS/blob/main/ETL/Sistema_BI_Transporte_P%C3%BAblico_Colectivo_AMSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ETL — Sistema BI Transporte Público Colectivo AMSS
## Viceministerio de Transporte (VMT) · OPAMSS

**Objetivo:** Extraer, limpiar y transformar los datos crudos del sistema de
transporte del AMSS desde GitHub, para generar los CSV limpios que se cargarán
al Data Warehouse en SQL Server.

**Flujo:**
Datos crudos (GitHub) - Extracción - Limpieza - Transformación - CSV limpios

| Tabla | Registros crudos | Registros limpios esperados |
|-------|-----------------|----------------------------|
| FACT_Parada_Servicio | 45,229 | ~38,200 |
| DIM_Parada | 9,354 | ~8,387 |
| DIM_Ruta | 1,188 | ~1,188 |
| DIM_Tarifa | 1,430 | ~1,426 |
| DIM_Municipio | 14 | 14 |
| DIM_Departamento | 15 | 15 |
| **TOTAL** | **57,230** | **~49,230** |

In [2]:
import pandas as pd
import numpy as np

print(f"pandas:  {pd.__version__}")
print(f"numpy:   {np.__version__}")
print("Librerías listas")

pandas:  2.2.2
numpy:   2.0.2
Librerías listas


## Sección 2 — Extracción
Carga de los 6 archivos CSV directamente desde el repositorio GitHub.

In [11]:
# Extracción desde GitHub

fact  = pd.read_csv("https://raw.githubusercontent.com/FernandoCasco/BI-Transporte-AMSS/refs/heads/main/Datos_crudos/FACT_Parada_Servicio.csv")
parada = pd.read_csv("https://raw.githubusercontent.com/FernandoCasco/BI-Transporte-AMSS/refs/heads/main/Datos_crudos/DIM_Parada.csv")
ruta   = pd.read_csv("https://raw.githubusercontent.com/FernandoCasco/BI-Transporte-AMSS/refs/heads/main/Datos_crudos/DIM_Ruta.csv")
tarifa = pd.read_csv("https://raw.githubusercontent.com/FernandoCasco/BI-Transporte-AMSS/refs/heads/main/Datos_crudos/DIM_Tarifa.csv")
mun    = pd.read_csv("https://raw.githubusercontent.com/FernandoCasco/BI-Transporte-AMSS/refs/heads/main/Datos_crudos/DIM_Municipio.csv")
dep    = pd.read_csv("https://raw.githubusercontent.com/FernandoCasco/BI-Transporte-AMSS/refs/heads/main/Datos_crudos/DIM_Departamento.csv")

print(" Registros crudos cargados:")
print(f"   FACT_Parada_Servicio: {len(fact):,}")
print(f"   DIM_Parada:           {len(parada):,}")
print(f"   DIM_Ruta:             {len(ruta):,}")
print(f"   DIM_Tarifa:           {len(tarifa):,}")
print(f"   DIM_Municipio:        {len(mun):,}")
print(f"   DIM_Departamento:     {len(dep):,}")
print(f"\n   TOTAL: {len(fact)+len(parada)+len(ruta)+len(tarifa)+len(mun)+len(dep):,} registros")

 Registros crudos cargados:
   FACT_Parada_Servicio: 45,229
   DIM_Parada:           9,354
   DIM_Ruta:             1,188
   DIM_Tarifa:           1,430
   DIM_Municipio:        14
   DIM_Departamento:     15

   TOTAL: 57,230 registros


## Sección 3 — Reporte inicial de calidad
Inspección de nulos y duplicados en cada tabla antes de limpiar.

In [12]:
# Reporte inicial de calidad
tablas = {
    "FACT_Parada_Servicio": fact,
    "DIM_Parada":           parada,
    "DIM_Ruta":             ruta,
    "DIM_Tarifa":           tarifa,
    "DIM_Municipio":        mun,
    "DIM_Departamento":     dep,
}

print("=" * 55)
print(f"{'Tabla':<25} {'Registros':>10} {'Nulos':>8} {'Duplicados':>10}")
print("=" * 55)

for nombre, df in tablas.items():
    total = len(df)
    nulos = df.isnull().sum().sum()
    dups  = df.duplicated().sum()
    print(f"{nombre:<25} {total:>10,} {nulos:>8,} {dups:>10,}")

print("=" * 55)
total_reg = sum(len(df) for df in tablas.values())
total_nul = sum(df.isnull().sum().sum() for df in tablas.values())
total_dup = sum(df.duplicated().sum() for df in tablas.values())
print(f"{'TOTAL':<25} {total_reg:>10,} {total_nul:>8,} {total_dup:>10,}")

Tabla                      Registros    Nulos Duplicados
FACT_Parada_Servicio          45,229    1,038          0
DIM_Parada                     9,354        3          0
DIM_Ruta                       1,188       10          0
DIM_Tarifa                     1,430        0          0
DIM_Municipio                     14        0          0
DIM_Departamento                  15        0          0
TOTAL                         57,230    1,051          0


## Detalle de nulos por columna

In [13]:
# Detalle de nulos por columna
for nombre, df in tablas.items():
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos) > 0:
        print(f"\n{nombre}:")
        for col, n in nulos.items():
            pct = n / len(df) * 100
            print(f"   {col:<25} {n:>6,} nulos  ({pct:.1f}%)")
    else:
        print(f"\n{nombre}: sin datos nulos")


FACT_Parada_Servicio:
   Frecuencia_Min             1,038 nulos  (2.3%)

DIM_Parada:
   Ruta                           1 nulos  (0.0%)
   Cod                            1 nulos  (0.0%)
   Parada_PGO                     1 nulos  (0.0%)

DIM_Ruta:
   Departamento                   1 nulos  (0.1%)
   Km_Recorrido                   1 nulos  (0.1%)
   Tarifa_USD                     8 nulos  (0.7%)

DIM_Tarifa: sin datos nulos

DIM_Municipio: sin datos nulos

DIM_Departamento: sin datos nulos


## Sección 4 — Limpieza y Transformación
Corrección de nulos, normalización de columnas y validación de datos por tabla.

In [14]:
# Limpieza y Transformación

# FACT_Parada_Servicio
fact_clean = fact.copy()

# Eliminar filas sin frecuencia (rutas sin servicio en ese período)
antes = len(fact_clean)
fact_clean = fact_clean.dropna(subset=["Frecuencia_Min"])
print(f"FACT_Parada_Servicio:")
print(f"  Nulos Frecuencia_Min eliminados: {antes - len(fact_clean):,}")

# Normalizar texto
fact_clean["Tipo_Unidad"]  = fact_clean["Tipo_Unidad"].str.upper().str.strip()
fact_clean["Tipo_Dia"]     = fact_clean["Tipo_Dia"].str.strip()
fact_clean["Periodo_Dia"]  = fact_clean["Periodo_Dia"].str.strip()
fact_clean["Municipio"]    = fact_clean["Municipio"].str.strip()
print(f"  Registros limpios: {len(fact_clean):,}")

# DIM_Parada
parada_clean = parada.copy()

antes = len(parada_clean)
parada_clean = parada_clean.dropna(subset=["Ruta", "Cod", "Parada_PGO"])
parada_clean["Municipio"] = parada_clean["NAM"].str.strip()
parada_clean["Cod"]       = parada_clean["Cod"].str.upper().str.strip()
print(f"\nDIM_Parada:")
print(f"  Nulos eliminados: {antes - len(parada_clean):,}")
print(f"  Registros limpios: {len(parada_clean):,}")

# DIM_Ruta
ruta_clean = ruta.copy()

antes = len(ruta_clean)
ruta_clean = ruta_clean.dropna(subset=["Departamento", "Km_Recorrido", "Tarifa_USD"])
ruta_clean["Sentido"]     = ruta_clean["Sentido"].str.upper().str.strip()
ruta_clean["Tipo_Unidad"] = ruta_clean["Tipo_Unidad"].str.upper().str.strip()
ruta_clean["Subtipo"]     = ruta_clean["Subtipo"].str.upper().str.strip()
print(f"\nDIM_Ruta:")
print(f"  Nulos eliminados: {antes - len(ruta_clean):,}")
print(f"  Registros limpios: {len(ruta_clean):,}")

# DIM_Tarifa
tarifa_clean = tarifa.copy()
tarifa_clean["Tipo_Unidad"] = tarifa_clean["Codigo_Ruta"].apply(
    lambda x: "AUTOBUS" if str(x).startswith("AB") else ("MICROBUS" if str(x).startswith("MB") else "DESCONOCIDO")
)
print(f"\nDIM_Tarifa:")
print(f"  Sin nulos — registros: {len(tarifa_clean):,}")

# DIM_Municipio
mun_clean = mun.copy()
mun_clean["Municipio"]   = mun_clean["Municipio"].str.strip()
mun_clean["Zona_AMSS"]   = mun_clean["Zona_AMSS"].str.strip()
print(f"\nDIM_Municipio:")
print(f"  Sin nulos — registros: {len(mun_clean):,}")

# DIM_Departamento
dep_clean = dep.copy()
print(f"\nDIM_Departamento:")
print(f"  Sin nulos — registros: {len(dep_clean):,}")

# RESUMEN
print(f"\n{'='*45}")
print(f"RESUMEN LIMPIEZA")
print(f"{'='*45}")
print(f"{'Tabla':<25} {'Crudo':>8} {'Limpio':>8} {'Eliminados':>10}")
print(f"{'-'*45}")
resumen = [
    ("FACT_Parada_Servicio", fact,   fact_clean),
    ("DIM_Parada",           parada, parada_clean),
    ("DIM_Ruta",             ruta,   ruta_clean),
    ("DIM_Tarifa",           tarifa, tarifa_clean),
    ("DIM_Municipio",        mun,    mun_clean),
    ("DIM_Departamento",     dep,    dep_clean),
]
total_crudo = total_limpio = 0
for nombre, crudo, limpio in resumen:
    eliminados = len(crudo) - len(limpio)
    total_crudo  += len(crudo)
    total_limpio += len(limpio)
    print(f"{nombre:<25} {len(crudo):>8,} {len(limpio):>8,} {eliminados:>10,}")
print(f"{'-'*45}")
print(f"{'TOTAL':<25} {total_crudo:>8,} {total_limpio:>8,} {total_crudo-total_limpio:>10,}")

FACT_Parada_Servicio:
  Nulos Frecuencia_Min eliminados: 1,038
  Registros limpios: 44,191

DIM_Parada:
  Nulos eliminados: 1
  Registros limpios: 9,353

DIM_Ruta:
  Nulos eliminados: 10
  Registros limpios: 1,178

DIM_Tarifa:
  Sin nulos — registros: 1,430

DIM_Municipio:
  Sin nulos — registros: 14

DIM_Departamento:
  Sin nulos — registros: 15

RESUMEN LIMPIEZA
Tabla                        Crudo   Limpio Eliminados
---------------------------------------------
FACT_Parada_Servicio        45,229   44,191      1,038
DIM_Parada                   9,354    9,353          1
DIM_Ruta                     1,188    1,178         10
DIM_Tarifa                   1,430    1,430          0
DIM_Municipio                   14       14          0
DIM_Departamento                15       15          0
---------------------------------------------
TOTAL                       57,230   56,181      1,049


## Sección 5 — Exportación de CSV limpios
Guardado de los 6 archivos procesados para subir a la carpeta Data_limpia/ del repositorio.

In [16]:
# Exportar CSV limpios

exportar = {
    "FACT_Parada_Servicio_clean": fact_clean,
    "DIM_Parada_clean":           parada_clean,
    "DIM_Ruta_clean":             ruta_clean,
    "DIM_Tarifa_clean":           tarifa_clean,
    "DIM_Municipio_clean":        mun_clean,
    "DIM_Departamento_clean":     dep_clean,
}

for nombre, df in exportar.items():
    df.to_csv(f"{nombre}.csv", index=False)
    print(f"✅ {nombre}.csv — {len(df):,} registros exportados")

✅ FACT_Parada_Servicio_clean.csv — 44,191 registros exportados
✅ DIM_Parada_clean.csv — 9,353 registros exportados
✅ DIM_Ruta_clean.csv — 1,178 registros exportados
✅ DIM_Tarifa_clean.csv — 1,430 registros exportados
✅ DIM_Municipio_clean.csv — 14 registros exportados
✅ DIM_Departamento_clean.csv — 15 registros exportados


## Resumen final del ETL

| Tabla | Registros crudos | Registros limpios | Eliminados |
|-------|-----------------|-------------------|------------|
| FACT_Parada_Servicio | 45,229 | 44,191 | 1,038 |
| DIM_Parada | 9,354 | 9,353 | 1 |
| DIM_Ruta | 1,188 | 1,178 | 10 |
| DIM_Tarifa | 1,430 | 1,430 | 0 |
| DIM_Municipio | 14 | 14 | 0 |
| DIM_Departamento | 15 | 15 | 0 |
| **TOTAL** | **57,230** | **56,181** | **1,049** |

**Problemas corregidos:**
- 1,038 nulos en `Frecuencia_Min` eliminados (rutas sin servicio nocturno dominical)
- 1 fila eliminada en `DIM_Parada` (Ruta, Cod y Parada_PGO nulos)
- 10 filas eliminadas en `DIM_Ruta` (Departamento, Km_Recorrido y Tarifa_USD nulos)
- Normalización de texto: UPPER + STRIP en columnas Sentido, Tipo_Unidad, Subtipo